# Goodreads Books Dataset Ingestion

Ingests the UCSD Goodreads Book Graph dataset (Mengting Wan et al.) into Unity Catalog as Delta tables: book metadata (ratings, page counts, publication dates, genres, popular shelves), authors, genre mappings, and per-review ratings. Raw files land in the `workspace.goodreads.raw` volume, parsed and flattened into queryable columns in the `workspace.goodreads` schema.

In [0]:
# Download Goodreads files to the volume
files = [
    ('goodreads_books.json.gz', 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz'),
    ('goodreads_reviews_dedup.json.gz', 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz'),
    ('goodreads_book_genres_initial.json.gz', 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_genres_initial.json.gz'),
    ('goodreads_book_authors.json.gz', 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_authors.json.gz'),
]

for fname, url in files:
    dest = f'/Volumes/workspace/goodreads/raw/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
        print(f'{fname} done')
    else:
        print(f'{fname} already exists; skipping')

goodreads_books.json.gz done
goodreads_reviews_dedup.json.gz done
goodreads_book_genres_initial.json.gz done
goodreads_book_authors.json.gz done


In [0]:
# Confirm downloads
for fname, _ in files:
    p = f'/Volumes/workspace/goodreads/raw/{fname}'
    sz = os.path.getsize(p) if os.path.exists(p) else 0
    print(fname, f'{sz/1e9:.2f} GB' if sz else 'missing')

goodreads_books.json.gz 2.08 GB
goodreads_reviews_dedup.json.gz 5.34 GB
goodreads_book_genres_initial.json.gz 0.02 GB
goodreads_book_authors.json.gz 0.02 GB


In [0]:
# Parse books metadata as Spark DataFrame
from pyspark.sql import functions as F
books_path = '/Volumes/workspace/goodreads/raw/goodreads_books.json.gz'
books_df = spark.read.json(books_path, multiLine=True)
print('Columns:', books_df.columns)
display(books_df.limit(5))

Columns: ['asin', 'authors', 'average_rating', 'book_id', 'country_code', 'description', 'edition_information', 'format', 'image_url', 'is_ebook', 'isbn', 'isbn13', 'kindle_asin', 'language_code', 'link', 'num_pages', 'popular_shelves', 'publication_day', 'publication_month', 'publication_year', 'publisher', 'ratings_count', 'series', 'similar_books', 'text_reviews_count', 'title', 'title_without_series', 'url', 'work_id']


asin,authors,average_rating,book_id,country_code,description,edition_information,format,image_url,is_ebook,isbn,isbn13,kindle_asin,language_code,link,num_pages,popular_shelves,publication_day,publication_month,publication_year,publisher,ratings_count,series,similar_books,text_reviews_count,title,title_without_series,url,work_id
,"List(List(604031, ))",4.00,5333265,US,,,Paperback,https://images.gr-assets.com/books/1310220028m/5333265.jpg,false,0312853122,9780312853129,,,https://www.goodreads.com/book/show/5333265-w-c-fields,256,"List(List(3, to-read), List(1, p), List(1, collection), List(1, w-c-fields), List(1, biography))",1,9,1984,St. Martin's Press,3,List(),List(),1,W.C. Fields: A Life on Film,W.C. Fields: A Life on Film,https://www.goodreads.com/book/show/5333265-w-c-fields,5400751
,"List(List(626222, ))",3.23,1333909,US,"Anita Diamant's international bestseller ""The Red Tent"" brilliantly re-created the ancient world of womanhood. Diamant brings her remarkable storytelling skills to ""Good Harbor"" -- offering insight to the precarious balance of marriage and career, motherhood and friendship in the world of modern women. The seaside town of Gloucester, Massachusetts is a place where the smell of the ocean lingers in the air and the rocky coast glistens in the Atlantic sunshine. When longtime Gloucester-resident Kathleen Levine is diagnosed with breast cancer, her life is thrown into turmoil. Frightened and burdened by secrets, she meets Joyce Tabachnik -- a freelance writer with literary aspirations -- and a once-in-a-lifetime friendship is born. Joyce has just bought a small house in Gloucester, where she hopes to write as well as vacation with her family. Like Kathleen, Joyce is at a fragile place in her life. A mutual love for books, humor, and the beauty of the natural world brings the two women together. They share their personal histories, and help each other to confront scars left by old emotional wounds. With her own trademark wisdom and humor, Diamant considers the nature, strength, and necessity of adult female friendship. ""Good Harbor"" examines the tragedy of loss, the insidious nature of family secrets, as well as the redemptive power of friendship.",Abridged,Audio CD,https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,false,0743509986,9780743509985,B000FC0PBC,,https://www.goodreads.com/book/show/1333909.Good_Harbor,,"List(List(2634, to-read), List(160, fiction), List(92, currently-reading), List(25, contemporary-fiction), List(22, owned), List(19, book-club), List(17, chick-lit), List(16, friendship), List(15, books-i-own), List(12, novel), List(12, general-fiction), List(11, anita-diamant), List(10, owned-books), List(10, contemporary), List(10, women), List(8, adult), List(7, default), List(7, favorites), List(7, didn-t-finish), List(6, own-it), List(6, library), List(6, new-england), List(6, bookclub), List(6, women-s-fiction), List(5, breast-cancer), List(5, beach-ocean), List(5, audio-book), List(5, historical-fiction), List(5, my-library), List(5, 2006), List(5, literature), List(5, audiobook), List(4, read-2015), List(4, diamant), List(4, audio-books), List(4, diamant-anita), List(4, adult-fiction), List(4, my-books), List(4, jewish), List(4, novels), List(4, library-book), List(4, 2002), List(3, read-2016), List(3, relationships), List(3, female-friendship), List(3, favorite-authors), List(3, kindle), List(3, have), List(3, strong-women), List(3, cancer), List(3, own-to-read), List(3, audiobooks), List(3, read-2011), List(3, audio), List(3, drama), List(3, read-2009), List(3, i-own), List(3, book-group), List(3, american), List(3, fiction-general), List(3, chicklit), List(3, book-club-books), List(2, hardback), List(2, my-personal-library), List(2, read-in-2017), List(2, read-2017), List(2, the-lewis-library), List(2, ficton), List(2, home), List(2, books), List(2, read-in-2015), List(2, listened), List(2, ebook), List(2, bookclub-books), List(2, to-buy), List(

In [0]:
# Inspect array/map fields to identify flattening pattern
books_df.printSchema()

# Show some nested/struct data for flattening
# There is no 'genres' or 'publication_date' column in this file. Use 'popular_shelves', 'publication_year', 'publication_month', 'publication_day' instead:
books_df.select('title','authors','popular_shelves','publication_year','publication_month','publication_day','average_rating','ratings_count','num_pages').limit(2).toPandas()

root
 |-- asin: string (nullable = true)
 |-- authors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author_id: string (nullable = true)
 |    |    |-- role: string (nullable = true)
 |-- average_rating: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- edition_information: string (nullable = true)
 |-- format: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- is_ebook: string (nullable = true)
 |-- isbn: string (nullable = true)
 |-- isbn13: string (nullable = true)
 |-- kindle_asin: string (nullable = true)
 |-- language_code: string (nullable = true)
 |-- link: string (nullable = true)
 |-- num_pages: string (nullable = true)
 |-- popular_shelves: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- count: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- pub

,title,authors,popular_shelves,publication_year,publication_month,publication_day,average_rating,ratings_count,num_pages
0,W.C. Fields: A Life on Film,"[{'author_id': '604031', 'role': ''}]","[{'count': '3', 'name': 'to-read'}, {'count': ...",1984,9,1,4.00,3,256
1,Good Harbor,"[{'author_id': '626222', 'role': ''}]","[{'count': '2634', 'name': 'to-read'}, {'count...",2001,10,1,3.23,10,


In [0]:
# Parse reviews as Spark DataFrame
reviews_path = '/Volumes/workspace/goodreads/raw/goodreads_reviews_dedup.json.gz'
reviews_df = spark.read.json(reviews_path, multiLine=True)
reviews_df.printSchema()
display(reviews_df.limit(5))

root
 |-- book_id: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- date_updated: string (nullable = true)
 |-- n_comments: long (nullable = true)
 |-- n_votes: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- read_at: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- user_id: string (nullable = true)



book_id,date_added,date_updated,n_comments,n_votes,rating,read_at,review_id,review_text,started_at,user_id
24375664,Fri Aug 25 13:55:02 -0700 2017,Mon Oct 09 08:55:59 -0700 2017,0,16,5,Sat Oct 07 00:00:00 -0700 2017,5cd416f3efc3f944fce4ce2db2290d5e,"Mind blowingly cool. Best science fiction I've read in some time. I just loved all the descriptions of the society of the future - how they lived in trees, the notion of owning property or even getting married was gone. How every surface was a screen. The undulations of how society responds to the Trisolaran threat seem surprising to me. Maybe its more the Chinese perspective, but I wouldn't have thought the ETO would exist in book 1, and I wouldn't have thought people would get so over-confident in our primitive fleet's chances given you have to think that with superior science they would have weapons - and defenses - that would just be as rifles to arrows once were. But the moment when Luo Ji won as a wallfacer was just too cool. I may have actually done a fist pump. Though by the way, if the Dark Forest theory is right - and I see no reason why it wouldn't be - we as a society should probably stop broadcasting so much signal out into the universe.",Sat Aug 26 00:00:00 -0700 2017,8842281e1d1347389f2ab93d60773d4d
18245960,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,1,28,5,Sat Aug 26 12:05:52 -0700 2017,dfdbb7b0eb5a7e4c26d59a937e2e5feb,"This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I've read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being ""reactionaries"", etc. It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. But what would happen if our SETI stations received a message - if we found someone was out there - and the person monitoring and answering the signal on our side was disillusioned? That part of the book was a bit dark - I would like to think human reaction to discovering alien civilization that is hostile would be more like Enders Game where we would band together. I did like how the book unveiled the Trisolaran culture through the game. It was a smart way to build empathy with them and also understand what they've gone through across so many centuries. And who know a 3 body problem was an unsolvable math problem? But I still don't get who made the game - maybe that will come in the next book. I loved this quote: ""In the long history of scientific progress, how many protons have been smashed apart in accelerators by physicists? How many neutrons and electrons? Probably no fewer than a hundred million. Every collision was probably the end of the civilizations and intelligences in a microcosmos. In fact, even in nature, the destruction of universes must be happening at every second--for example, through the decay of neutrons. Also, a high-energy cosmic ray entering the atmosphere may destroy thousands of such miniature universes....""",Tue Aug 15 13:23:18 -0700 2017,8842281e1d1347389f2ab93d60773d4d
6392944,Mon Jul 24 02:48:17 -0700 2017,Sun Jul 30 09:28:03 -0700 2017,0,6,3,Tue Jul 25 00:00:00 -0700 2017,5e212a62bced17b4dbe41150e5bb9037,"I haven't read a fun mystery book in a while and not sure I've ever read Poirot. Was looking for a fun read set in France while I was on holiday there and this didn't disappoint! Fast paced and good mystery. One that struck me was how similar Poirot is to Sherlock. They are both detectives, 

In [0]:
# Parse genres mapping file
genres_path = '/Volumes/workspace/goodreads/raw/goodreads_book_genres_initial.json.gz'
genres_df = spark.read.json(genres_path, multiLine=True)
genres_df.printSchema()
display(genres_df.limit(5))

root
 |-- book_id: string (nullable = true)
 |-- genres: struct (nullable = true)
 |    |-- history, historical fiction, biography: long (nullable = true)



book_id,genres
5333265,List(1)
1333909,List(5)
7327624,List(null)
6066819,List(null)
287140,List(null)


In [0]:
# Parse author metadata
authors_path = '/Volumes/workspace/goodreads/raw/goodreads_book_authors.json.gz'
authors_df = spark.read.json(authors_path, multiLine=True)
authors_df.printSchema()
display(authors_df.limit(5))

root
 |-- author_id: string (nullable = true)
 |-- average_rating: string (nullable = true)
 |-- name: string (nullable = true)
 |-- ratings_count: string (nullable = true)
 |-- text_reviews_count: string (nullable = true)



author_id,average_rating,name,ratings_count,text_reviews_count
604031,3.98,Ronald J. Fields,49,7
626222,4.08,Anita Diamant,546796,28716
10333,3.92,Barbara Hambly,122118,5075
9212,3.68,Jennifer Weiner,888522,36262
149918,3.82,Nigel Pennick,1740,96


In [0]:
# Flatten books into queryable columns and write as a Delta table
from pyspark.sql import functions as F

books_flat = books_df.select(
    F.col("book_id"),
    F.col("work_id"),
    F.col("title"),
    F.col("title_without_series"),
    F.expr("transform(authors, a -> a.author_id)").alias("author_ids"),
    F.expr("try_element_at(authors, 1).author_id").alias("primary_author_id"),
    F.expr("TRY_CAST(average_rating AS DOUBLE)").alias("average_rating"),
    F.expr("TRY_CAST(ratings_count AS BIGINT)").alias("ratings_count"),
    F.expr("TRY_CAST(text_reviews_count AS BIGINT)").alias("text_reviews_count"),
    F.expr("TRY_CAST(num_pages AS INT)").alias("num_pages"),
    F.expr("TRY_CAST(publication_year AS INT)").alias("publication_year"),
    F.expr("TRY_CAST(publication_month AS INT)").alias("publication_month"),
    F.expr("TRY_CAST(publication_day AS INT)").alias("publication_day"),
    F.col("publisher"),
    F.col("language_code"),
    F.col("format"),
    F.col("is_ebook"),
    F.col("description"),
    F.expr("transform(popular_shelves, s -> struct(s.name as name, cast(s.count as long) as count))").alias("popular_shelves"),
).withColumn(
    "publication_date",
    F.when(
        F.col("publication_year").isNotNull()
        & F.col("publication_month").isNotNull()
        & F.col("publication_day").isNotNull(),
        F.expr("TRY_CAST(concat_ws('-', lpad(publication_year, 4, '0'), lpad(publication_month, 2, '0'), lpad(publication_day, 2, '0')) AS DATE)"),
    ),
)

books_flat.write.mode("overwrite").saveAsTable("workspace.goodreads.books")
print("books rows:", spark.table("workspace.goodreads.books").count())
display(spark.table("workspace.goodreads.books").limit(5))

books rows: 2360655


book_id,work_id,title,title_without_series,author_ids,primary_author_id,average_rating,ratings_count,text_reviews_count,num_pages,publication_year,publication_month,publication_day,publisher,language_code,format,is_ebook,description,popular_shelves,publication_date
5333265,5400751,W.C. Fields: A Life on Film,W.C. Fields: A Life on Film,List(604031),604031,4.0,3,1,256,1984,9,1,St. Martin's Press,,Paperback,false,,"List(List(to-read, 3), List(p, 1), List(collection, 1), List(w-c-fields, 1), List(biography, 1))",1984-09-01
1333909,1323437,Good Harbor,Good Harbor,List(626222),626222,3.23,10,6,null,2001,10,1,Simon & Schuster Audio,,Audio CD,false,"Anita Diamant's international bestseller ""The Red Tent"" brilliantly re-created the ancient world of womanhood. Diamant brings her remarkable storytelling skills to ""Good Harbor"" -- offering insight to the precarious balance of marriage and career, motherhood and friendship in the world of modern women. The seaside town of Gloucester, Massachusetts is a place where the smell of the ocean lingers in the air and the rocky coast glistens in the Atlantic sunshine. When longtime Gloucester-resident Kathleen Levine is diagnosed with breast cancer, her life is thrown into turmoil. Frightened and burdened by secrets, she meets Joyce Tabachnik -- a freelance writer with literary aspirations -- and a once-in-a-lifetime friendship is born. Joyce has just bought a small house in Gloucester, where she hopes to write as well as vacation with her family. Like Kathleen, Joyce is at a fragile place in her life. A mutual love for books, humor, and the beauty of the natural world brings the two women together. They share their personal histories, and help each other to confront scars left by old emotional wounds. With her own trademark wisdom and humor, Diamant considers the nature, strength, and necessity of adult female friendship. ""Good Harbor"" examines the tragedy of loss, the insidious nature of family secrets, as well as the redemptive power of friendship.","List(List(to-read, 2634), List(fiction, 160), List(currently-reading, 92), List(contemporary-fiction, 25), List(owned, 22), List(book-club, 19), List(chick-lit, 17), List(friendship, 16), List(books-i-own, 15), List(novel, 12), List(general-fiction, 12), List(anita-diamant, 11), List(owned-books, 10), List(contemporary, 10), List(women, 10), List(adult, 8), List(default, 7), List(favorites, 7), List(didn-t-finish, 7), List(own-it, 6), List(library, 6), List(new-england, 6), List(bookclub, 6), List(women-s-fiction, 6), List(breast-cancer, 5), List(beach-ocean, 5), List(audio-book, 5), List(historical-fiction, 5), List(my-library, 5), List(2006, 5), List(literature, 5), List(audiobook, 5), List(read-2015, 4), List(diamant, 4), List(audio-books, 4), List(diamant-anita, 4), List(adult-fiction, 4), List(my-books, 4), List(jewish, 4), List(novels, 4), List(library-book, 4), List(2002, 4), List(read-2016, 3), List(relationships, 3), List(female-friendship, 3), List(favorite-authors, 3), List(kindle, 3), List(have, 3), List(strong-women, 3), List(cancer, 3), List(own-to-read, 3), List(audiobooks, 3), List(read-2011, 3), List(audio, 3), List(drama, 3), List(read-2009, 3), List(i-own, 3), List(book-group, 3), List(american, 3), List(fiction-general, 3), List(chicklit, 3), List(book-club-books, 3), List(hardback, 2), List(my-personal-library, 2), List(read-in-2017, 2), List(read-2017, 2), List(the-lewis-library, 2), List(ficton, 2), List(home, 2), List(books, 2), List(read-in-2015, 2), List(listened, 2), List(ebook, 2), List(bookclub-books, 2), List(to-buy, 2), List(my-home-library, 2), List(summer-reading, 2), List(home-bookshelf, 2), List(family, 2), List(tbi-book-club, 2), List(wish-list, 2), List(own-book, 2), List(unfinished, 2), List(fiction-contemporary, 2), List(adult-books, 2), List(books-i-have, 2), List(paperback, 2), List(books-in-my-library, 2), List(summer-reads, 2), List(read-in-2011, 2), List(read-in-2010, 2), List(gave-up, 2), List(book

In [0]:
# Flatten authors and write as a Delta table
authors_flat = authors_df.select(
    F.col("author_id"),
    F.col("name"),
    F.col("average_rating").cast("double").alias("average_rating"),
    F.col("ratings_count").cast("long").alias("ratings_count"),
    F.col("text_reviews_count").cast("long").alias("text_reviews_count"),
)
authors_flat.write.mode("overwrite").saveAsTable("workspace.goodreads.authors")
print("authors rows:", spark.table("workspace.goodreads.authors").count())
display(spark.table("workspace.goodreads.authors").limit(5))

authors rows: 829529


author_id,name,average_rating,ratings_count,text_reviews_count
604031,Ronald J. Fields,3.98,49,7
626222,Anita Diamant,4.08,546796,28716
10333,Barbara Hambly,3.92,122118,5075
9212,Jennifer Weiner,3.68,888522,36262
149918,Nigel Pennick,3.82,1740,96


In [0]:
# Genres file: one row per book, 'genres' maps genre name -> count. Explode to (book_id, genre, shelf_count)
genres_map = spark.read.schema("book_id STRING, genres MAP<STRING, BIGINT>").json(
    "/Volumes/workspace/goodreads/raw/goodreads_book_genres_initial.json.gz"
)
genres_flat = genres_map.select("book_id", F.explode("genres").alias("genre", "shelf_count"))
genres_flat.write.mode("overwrite").saveAsTable("workspace.goodreads.book_genres")
print("book_genres rows:", spark.table("workspace.goodreads.book_genres").count())
display(spark.table("workspace.goodreads.book_genres").limit(5))

book_genres rows: 5042542


book_id,genre,shelf_count
5333265,"history, historical fiction, biography",1
1333909,fiction,219
1333909,"history, historical fiction, biography",5
7327624,"fantasy, paranormal",31
7327624,fiction,8


In [0]:
# Reviews: keep core fields, parse the timestamp strings, write as a Delta table
# Goodreads timestamps look like 'Fri Aug 25 13:55:02 -0700 2017' (SimpleDateFormat style).
# Use the legacy parser, which understands EEE/zzz patterns.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

reviews_flat = reviews_df.select(
    F.col("review_id"),
    F.col("book_id"),
    F.col("user_id"),
    F.col("rating"),
    F.col("review_text"),
    F.col("n_votes"),
    F.col("n_comments"),
    F.col("date_added"),
    F.col("date_updated"),
).withColumn(
    # try_to_timestamp returns NULL for malformed/out-of-range dates (e.g. year 1092)
    "date_added_ts",
    F.expr("try_to_timestamp(date_added, 'EEE MMM dd HH:mm:ss zzz yyyy')"),
)

reviews_flat.write.mode("overwrite").saveAsTable("workspace.goodreads.reviews")
print("reviews rows:", spark.table("workspace.goodreads.reviews").count())

reviews rows: 15739967


In [0]:
# Verify final tables: row counts and total on-disk size
from pyspark.sql import functions as F

total_bytes = 0
for t in ["books", "authors", "book_genres", "reviews"]:
    d = spark.sql(f"DESCRIBE DETAIL workspace.goodreads.{t}").collect()[0]
    n = spark.sql(f"SELECT COUNT(*) AS c FROM workspace.goodreads.{t}").collect()[0]["c"]
    total_bytes += d["sizeInBytes"]
    print(f"{t}: {n:,} rows, {d['sizeInBytes']/1e9:.2f} GB on disk")

print(f"\nTOTAL: {total_bytes/1e9:.2f} GB — 1GB+ requirement {'MET' if total_bytes > 1e9 else 'NOT MET'}")

books: 2,360,655 rows, 1.56 GB on disk
authors: 829,529 rows, 0.01 GB on disk
book_genres: 5,042,542 rows, 0.02 GB on disk
reviews: 15,739,967 rows, 5.31 GB on disk

TOTAL: 6.91 GB — 1GB+ requirement MET
